# External emotion evaluation — test split only

Breaks the internal-encoder circularity flagged by reviewer: evaluates generated videos with **classifiers never involved in training or model selection**.

**Protocol:**
1. For each fine-tuned checkpoint (baseline + ablation configs), generate full-length videos from **test split** audio.
2. Run independent classifiers — all trained on non-RAVDESS data — on the generated frames.
3. Compare external accuracy / F1 against the ground-truth emotion label.
4. Report Δ external-F1 vs baseline with statistical significance.

**External models:**
- **Video (independent classifiers)**:
  - `dima806/facial_emotions_image_detection` — ViT trained on FER-derived corpus.
  - `trpakov/vit-face-expression` — ViT trained on a separate FER variant.
  Both never seen by training/model selection in 02–06.
- **Audio (optional dim.)**: `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` — MSP-Podcast dimensional emotion (arousal/dominance/valence) for cross-modal sanity check.

Neither was used at any stage of training or model selection.

In [16]:
!nvidia-smi

Sun May  3 00:17:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.142                Driver Version: 580.142        CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:0B:00.0  On |                  N/A |
| 30%   40C    P8             29W /  350W |    2166MiB /  24576MiB |     35%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
!pip install -q transformers librosa scipy scikit-learn Pillow

In [18]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/Wav2Lip")

import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from PIL import Image
from scipy import stats
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    AutoFeatureExtractor, AutoModel,
)

from models.wav2lip import Wav2Lip as Wav2LipModel

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
WAV2LIP_CKPT = "/content/Wav2Lip/checkpoints/wav2lip.pth"

ABLATION_DIR = Path("/content/wav2lip_loss_ablation_4emo")
ORIG_DIR = Path("/content/wav2lip_finetuned_4emo")
OUT_DIR = Path("/content/external_eval_results_4emo")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 4-emotion setup (aligned with 01_data_preprocessing EMOTION_TO_IDX and
#     02_train_encoders_4emotions / 04a_wav2lip_loss_ablation) ---
# RAVDESS raw idx:  neutral=0, calm=1, happy=2, sad=3, angry=4, fearful=5, disgust=6, surprised=7
EMOTIONS = ["happy", "sad", "angry", "disgust"]
EMOTION_NAME_TO_RAW_IDX = {
    "neutral": 0, "calm": 1, "happy": 2, "sad": 3,
    "angry": 4, "fearful": 5, "disgust": 6, "surprised": 7,
}
INCLUDE = {EMOTION_NAME_TO_RAW_IDX[e] for e in EMOTIONS}       # raw RAVDESS indices we keep
EXCLUDE = set(EMOTION_NAME_TO_RAW_IDX.values()) - INCLUDE      # raw indices we drop (neutral, calm)
REMAP = {EMOTION_NAME_TO_RAW_IDX[e]: i for i, e in enumerate(EMOTIONS)}  # raw -> our 0..NUM_EMO-1
NUM_EMO = len(EMOTIONS)

print(f"Device: {DEVICE}")
print(f"Emotions ({NUM_EMO}): {EMOTIONS}")
print(f"Include RAVDESS idx: {sorted(INCLUDE)}  |  Exclude: {sorted(EXCLUDE)}")
print(f"REMAP: {REMAP}")

Device: cuda
Emotions (4): ['happy', 'sad', 'angry', 'disgust']
Include RAVDESS idx: [2, 3, 4, 6]  |  Exclude: [0, 1, 5, 7]
REMAP: {2: 0, 3: 1, 4: 2, 6: 3}


## Checkpoints to evaluate

Edit this list to match the runs you want to report. Each entry is `(display_name, checkpoint_path)`.

In [19]:
# Two-stage external evaluation:
#   1. 04b loss-family ablation (different loss types, one weight each) — externally
#      validates which loss family generalizes beyond the internal TimeSformer.
#   2. 04 weight sweep (winner of 04b at multiple scales) — externally validates
#      the chosen weight given the loss family.
# Path.is_file() filter at the bottom silently skips missing checkpoints, so
# you can run any subset that's present on /content.
CHECKPOINTS = [
    # --- Loss-family ablation (04a_wav2lip_loss_ablation.ipynb) ---
    ("abl-baseline", ABLATION_DIR / "abl-baseline" / "wav2lip.pth"),
    ("abl-cos-only", ABLATION_DIR / "abl-cos-only" / "wav2lip.pth"),
    ("abl-ce-only",  ABLATION_DIR / "abl-ce-only"  / "wav2lip.pth"),
    ("abl-ce-cos",   ABLATION_DIR / "abl-ce-cos"   / "wav2lip.pth"),
    ("abl-ce-kl",    ABLATION_DIR / "abl-ce-kl"    / "wav2lip.pth"),
    # --- 04 weight sweep on CE+KL family (the previously-presumed winner) ---
    ("baseline",     ORIG_DIR / "wav2lip-baseline" / "wav2lip.pth"),
    ("cekl-01",      ORIG_DIR / "wav2lip-cekl-01"  / "wav2lip.pth"),
    ("cekl-025",     ORIG_DIR / "wav2lip-cekl-025" / "wav2lip.pth"),
    ("cekl-04",      ORIG_DIR / "wav2lip-cekl-04"  / "wav2lip.pth"),
    ("cekl-05",      ORIG_DIR / "wav2lip-cekl-05"  / "wav2lip.pth"),
]
CHECKPOINTS = [(n, p) for n, p in CHECKPOINTS if Path(p).is_file()]
print(f"Evaluating {len(CHECKPOINTS)} checkpoints:")
for n, p in CHECKPOINTS:
    print(f"  {n:20s} -> {p}")

Evaluating 10 checkpoints:
  abl-baseline         -> /content/wav2lip_loss_ablation_4emo/abl-baseline/wav2lip.pth
  abl-cos-only         -> /content/wav2lip_loss_ablation_4emo/abl-cos-only/wav2lip.pth
  abl-ce-only          -> /content/wav2lip_loss_ablation_4emo/abl-ce-only/wav2lip.pth
  abl-ce-cos           -> /content/wav2lip_loss_ablation_4emo/abl-ce-cos/wav2lip.pth
  abl-ce-kl            -> /content/wav2lip_loss_ablation_4emo/abl-ce-kl/wav2lip.pth
  baseline             -> /content/wav2lip_finetuned_4emo/wav2lip-baseline/wav2lip.pth
  cekl-01              -> /content/wav2lip_finetuned_4emo/wav2lip-cekl-01/wav2lip.pth
  cekl-025             -> /content/wav2lip_finetuned_4emo/wav2lip-cekl-025/wav2lip.pth
  cekl-04              -> /content/wav2lip_finetuned_4emo/wav2lip-cekl-04/wav2lip.pth
  cekl-05              -> /content/wav2lip_finetuned_4emo/wav2lip-cekl-05/wav2lip.pth


## Load external classifier (video — FER-trained ViT, independent of training)

`dima806/facial_emotions_image_detection` outputs 7 classes: `angry, disgust, fear, happy, neutral, sad, surprise`. The mapping to our target set is resolved automatically by name (with common-variant aliases, e.g. `fear`↔`fearful`, `surprise`↔`surprised`), so this cell works regardless of whether `EMOTIONS` is the 3-class or 6-class list defined in cell 2.

Emotions that don't have an external counterpart (e.g., `calm`, `neutral`) would raise a clear error — by design, to catch accidental mismatches early.

In [ ]:
# Two independent external video FER classifiers — neither shared weights with
# 02 training, 03 utils, 04a loss ablation, or 06 finetune. Validating on both
# guards against single-classifier bias (e.g., dima806 never predicts disgust).
# Picked from external_screening.ipynb on real RAVDESS train+val (n=640).
# motheecreator: F1 0.65 / Rajaram1996: F1 0.58.
# Dropped: dima806 (F1 0.39, F1_disgust = 0), trpakov (F1 0.53, F1_disgust = 0.11).
# Both retained externals reach F1_disgust > 0.65 → can distinguish all 4 emotions.
EXTERNAL_VIDEO_MODELS = [
    ("motheecreator", "motheecreator/vit-Facial-Expression-Recognition"),
    ("Rajaram1996",   "Rajaram1996/FacialEmoRecog"),
]

# Robust name resolver — handles short/long forms across HF forks.
_NAME_ALIASES = {
    "happy":     ["happy", "happiness"],
    "sad":       ["sad", "sadness"],
    "angry":     ["angry", "anger"],
    "fearful":   ["fear", "fearful"],
    "disgust":   ["disgust", "disgusted"],
    "surprised": ["surprise", "surprised"],
}


def _resolve_target_ids(label2id_raw, ext_name):
    """Map our EMOTIONS list to indices in the external classifier label space."""
    out = {}
    for our in EMOTIONS:
        for alias in _NAME_ALIASES.get(our, [our]):
            if alias in label2id_raw:
                out[our] = label2id_raw[alias]
                break
        else:
            raise KeyError(
                f"External '{ext_name}' has no label for '{our}'. "
                f"Available: {sorted(label2id_raw)}"
            )
    return out


EXTERNALS = {}
for ext_name, model_id in EXTERNAL_VIDEO_MODELS:
    proc = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForImageClassification.from_pretrained(model_id).to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    id2label = model.config.id2label
    label2id_raw = {v.lower(): k for k, v in id2label.items()}
    target_ids = _resolve_target_ids(label2id_raw, ext_name)
    EXTERNALS[ext_name] = {
        "model_id":   model_id,
        "proc":       proc,
        "model":      model,
        "id2label":   id2label,
        "target_ids": target_ids,
    }
    print(f"[{ext_name}] {model_id}")
    print(f"  labels:        {id2label}")
    print(f"  target→ext id: {target_ids}")

## Test dataset — full-length videos

Generates all 32 preprocessed frames per test sample for richer external-classifier signal (vs. the random 5-frame windows used in training).

In [21]:
IMG_SIZE = 96
MEL_STEP = 16
SR = 16000
FPS = 25


def wav_to_mel(wav_path, sr=SR):
    y, _ = librosa.load(wav_path, sr=sr)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=80, hop_length=200, win_length=800, fmin=55, fmax=7600)
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


class Wav2LipTestDataset(Dataset):
    """Full-length, deterministic test samples for external evaluation."""

    def __init__(self, metadata_path, split="test"):
        with open(metadata_path) as f:
            data = json.load(f)
        self.samples = [
            s for s in data
            if s["split"] == split and s["emotion_idx"] not in EXCLUDE
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav, _ = torchaudio.load(s["audio_path"])
        audio_1d = wav.squeeze(0)
        mel = wav_to_mel(s["audio_path"])

        frames = np.load(s["frames_path"]).astype(np.float32) / 255.0  # (T, H, W, 3)
        T = frames.shape[0]

        gt = torch.from_numpy(frames).permute(0, 3, 1, 2)
        if gt.shape[2] != IMG_SIZE or gt.shape[3] != IMG_SIZE:
            gt = F.interpolate(gt, size=(IMG_SIZE, IMG_SIZE), mode="bilinear",
                                align_corners=False)

        masked = gt.clone()
        masked[:, :, IMG_SIZE // 2:, :] = 0.0
        # Deterministic reference frame: use first frame
        ref = gt[0:1].expand(T, -1, -1, -1)
        face_input = torch.cat([ref, masked], dim=1)

        # Trim mel to exactly T frames
        need = MEL_STEP * T
        if mel.shape[1] < need:
            mel = np.pad(mel, ((0, 0), (0, need - mel.shape[1])))
        else:
            mel = mel[:, :need]
        mel_chunks = [
            torch.from_numpy(mel[:, t * MEL_STEP:(t + 1) * MEL_STEP]).unsqueeze(0)
            for t in range(T)
        ]
        mel_tensor = torch.stack(mel_chunks, dim=0)  # (T, 1, 80, 16)

        return {
            "sample_id": s["sample_id"],
            "mel": mel_tensor,
            "face_input": face_input,
            "gt": gt,
            "audio": audio_1d,
            "emotion": REMAP[s["emotion_idx"]],
            "n_frames": T,
        }


test_ds = Wav2LipTestDataset(METADATA, split="test")
print(f"Test samples: {len(test_ds)}")
from collections import Counter
print("Per-emotion counts:",
      dict(Counter(EMOTIONS[s["emotion"]] for s in test_ds)))

Test samples: 96
Per-emotion counts: {'happy': 24, 'sad': 24, 'angry': 24, 'disgust': 24}


In [22]:
def load_wav2lip(ckpt_path, device):
    model = Wav2LipModel()
    try:
        ckpt = torch.load(WAV2LIP_CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(WAV2LIP_CKPT, map_location="cpu")
    state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    # Overlay fine-tuned weights
    try:
        ft = torch.load(ckpt_path, map_location=device, weights_only=True)
    except TypeError:
        ft = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ft, strict=False)
    return model.to(device).eval()


@torch.no_grad()
def generate_full_video(model, sample):
    """Returns generated video tensor (T, 3, 96, 96) on CPU."""
    mel = sample["mel"].unsqueeze(0).to(DEVICE)  # (1, T, 1, 80, 16)
    face_in = sample["face_input"].unsqueeze(0).to(DEVICE)
    T = mel.shape[1]
    gens = []
    for t in range(T):
        g = model(mel[:, t], face_in[:, t])
        gens.append(g.squeeze(0).cpu())
    return torch.stack(gens, dim=0).clamp(0, 1)

## Per-sample external video prediction

Strategy: classify every 4th generated frame with the external FER ViT, average softmax probabilities, then take argmax **restricted to our target classes** for per-sample prediction. Per-frame full 7-class probabilities are also kept for confusion analysis.

In [23]:
FRAME_STRIDE = 4           # classify 1 of every N generated frames
EXT_BATCH = 32             # batch size for external classifier


@torch.no_grad()
def external_predict_video(gen_video, ext_name):
    """gen_video: (T, 3, 96, 96) in [0, 1]. Returns mean_probs over sampled frames,
    sample-level argmax over our EMOTIONS, and per-frame restricted argmax list.
    `ext_name` selects an entry in the EXTERNALS dict from cell 6."""
    ext = EXTERNALS[ext_name]
    frames = gen_video[::FRAME_STRIDE]
    pil_frames = [
        Image.fromarray((f.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
        for f in frames
    ]
    all_probs = []
    for i in range(0, len(pil_frames), EXT_BATCH):
        chunk = pil_frames[i:i + EXT_BATCH]
        inputs = ext["proc"](chunk, return_tensors="pt").to(DEVICE)
        logits = ext["model"](**inputs).logits
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu())
    probs = torch.cat(all_probs, dim=0)
    mean_probs = probs.mean(dim=0)

    tgt_indices = [ext["target_ids"][e] for e in EMOTIONS]
    restricted = probs[:, tgt_indices]
    per_frame_labels = restricted.argmax(dim=1).tolist()
    sample_label = int(mean_probs[tgt_indices].argmax().item())
    return mean_probs.numpy(), sample_label, per_frame_labels

In [ ]:
def evaluate_checkpoint(name, ckpt_path):
    """Returns {ext_name: DataFrame} so a single Wav2Lip generation pass feeds
    every external classifier. Cuts external compute roughly in half vs running
    each external as a separate full pass."""
    model = load_wav2lip(ckpt_path, DEVICE)
    rows_per_ext = {ext_name: [] for ext_name in EXTERNALS}
    for i in tqdm(range(len(test_ds)), desc=f"[{name}]", leave=False):
        sample = test_ds[i]
        gen = generate_full_video(model, sample)
        for ext_name in EXTERNALS:
            mean_probs, sample_label, per_frame = external_predict_video(gen, ext_name)
            rows_per_ext[ext_name].append({
                "sample_id":     sample["sample_id"],
                "emotion_true":  sample["emotion"],
                "ext_pred":      sample_label,
                "ext_probs":     mean_probs.tolist(),
                "ext_per_frame": per_frame,
            })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    out = {}
    for ext_name, rows in rows_per_ext.items():
        df = pd.DataFrame(rows)
        df.to_json(OUT_DIR / f"{name}__{ext_name}.json", orient="records")
        out[ext_name] = df
    return out


# per_model[checkpoint_name][ext_name] = DataFrame
per_model = {}
for name, path in CHECKPOINTS:
    print(f"\n=== Evaluating {name} ===")
    per_model[name] = evaluate_checkpoint(name, path)
print("\nAll checkpoints evaluated against", list(EXTERNALS), "externals.")

In [ ]:
def model_metrics(df):
    y = df["emotion_true"].to_numpy()
    p = df["ext_pred"].to_numpy()
    acc = accuracy_score(y, p)
    f1_macro = f1_score(y, p, labels=list(range(NUM_EMO)), average="macro", zero_division=0)
    per_f1 = f1_score(y, p, labels=list(range(NUM_EMO)), average=None, zero_division=0)
    return acc, f1_macro, per_f1


summary_per_ext = {}
for ext_name in EXTERNALS:
    rows = []
    for name in [n for n, _ in CHECKPOINTS]:
        acc, f1m, per = model_metrics(per_model[name][ext_name])
        rows.append({
            "config": name,
            "ext_accuracy": acc,
            "ext_F1_macro": f1m,
            **{f"ext_F1_{EMOTIONS[i]}": per[i] for i in range(NUM_EMO)},
        })
    df = pd.DataFrame(rows)
    base_f1 = df.loc[df["config"] == "baseline", "ext_F1_macro"].iloc[0]
    df["Δ_F1_vs_baseline"] = df["ext_F1_macro"] - base_f1
    df = df.sort_values("ext_F1_macro", ascending=False).reset_index(drop=True)
    summary_per_ext[ext_name] = df
    print(f"\n=== EXTERNAL ({EXTERNALS[ext_name]['model_id']}) on test split ===")
    print(df.to_string(index=False))
    df.to_csv(OUT_DIR / f"summary__{ext_name}.csv", index=False)

In [ ]:
# McNemar vs baseline — did external emotion recognition meaningfully improve?
# Run separately for each external so each contributes its own significance test.
mcnemar_per_ext = {}
for ext_name in EXTERNALS:
    base_df = per_model["baseline"][ext_name].set_index("sample_id")
    rows = []
    for name, _ in CHECKPOINTS:
        if name == "baseline":
            continue
        other = per_model[name][ext_name].set_index("sample_id")
        shared = base_df.index.intersection(other.index)
        y = base_df.loc[shared, "emotion_true"].to_numpy()
        b_ok = (base_df.loc[shared, "ext_pred"].to_numpy() == y)
        e_ok = (other.loc[shared, "ext_pred"].to_numpy() == y)
        n01 = int((b_ok & ~e_ok).sum())
        n10 = int((~b_ok & e_ok).sum())
        chi2 = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
        p_val = 1 - stats.chi2.cdf(chi2, df=1) if (n01 + n10) > 0 else 1.0
        rows.append({
            "config":               name,
            "correct_b→w":          n01,
            "wrong_b→c":            n10,
            "McNemar χ²":           chi2,
            "McNemar p":            p_val,
            "significant (p<0.05)": p_val < 0.05,
        })
    df = pd.DataFrame(rows)
    mcnemar_per_ext[ext_name] = df
    print(f"\n=== McNemar vs baseline ({EXTERNALS[ext_name]['model_id']}) ===")
    print(df.to_string(index=False))
    df.to_csv(OUT_DIR / f"mcnemar__{ext_name}.csv", index=False)


# Agreement check: which configs are externally-better on BOTH classifiers?
print("\n=== Cross-external agreement (Δ F1 vs baseline) ===")
agree_rows = []
for name, _ in CHECKPOINTS:
    if name == "baseline":
        continue
    deltas = {ext: float(summary_per_ext[ext]
                          .loc[summary_per_ext[ext]["config"] == name, "Δ_F1_vs_baseline"].iloc[0])
              for ext in EXTERNALS}
    agree_rows.append({"config": name, **{f"Δ_{ext}": d for ext, d in deltas.items()},
                        "both_positive": all(d > 0 for d in deltas.values()),
                        "both_negative": all(d < 0 for d in deltas.values())})
agree_df = pd.DataFrame(agree_rows)
print(agree_df.to_string(index=False))
agree_df.to_csv(OUT_DIR / "agreement.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Per-external bar plots: macro F1 + per-emotion F1
for ext_name in EXTERNALS:
    sdf = summary_per_ext[ext_name]
    base_f1 = sdf.loc[sdf["config"] == "baseline", "ext_F1_macro"].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    names = sdf["config"].tolist()

    axes[0].bar(names, sdf["ext_F1_macro"], color="steelblue", alpha=0.85)
    axes[0].axhline(base_f1, color="red", ls="--", label=f"baseline={base_f1:.3f}")
    axes[0].set_title(f"External macro-F1 — {EXTERNALS[ext_name]['model_id']}")
    axes[0].set_ylabel("F1")
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].legend()

    x = np.arange(NUM_EMO)
    w = 0.8 / len(names)
    for i, n in enumerate(names):
        vals = [sdf.loc[sdf["config"] == n, f"ext_F1_{e}"].iloc[0] for e in EMOTIONS]
        axes[1].bar(x + (i - len(names) / 2 + 0.5) * w, vals, w, label=n)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(EMOTIONS)
    axes[1].set_ylabel("External F1")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"Per-emotion F1 — {ext_name}")
    axes[1].legend(fontsize=7, ncol=len(names))
    plt.tight_layout()
    plt.show()


# Confusion matrices: baseline vs best-by-external (side-by-side per external)
print("\n=== External confusion matrices (rows=true, cols=predicted) ===")
for ext_name in EXTERNALS:
    sdf = summary_per_ext[ext_name]
    best_name = sdf.iloc[0]["config"]
    cm_pairs = []
    if "baseline" in per_model and best_name != "baseline":
        cm_pairs.append(("baseline", per_model["baseline"][ext_name]))
    cm_pairs.append((best_name, per_model[best_name][ext_name]))

    print(f"\n--- {ext_name} ({EXTERNALS[ext_name]['model_id']}) ---")
    for title, dfm in cm_pairs:
        cm = confusion_matrix(dfm["emotion_true"], dfm["ext_pred"], labels=list(range(NUM_EMO)))
        print(f"\n  {title}")
        print(pd.DataFrame(cm, index=EMOTIONS, columns=EMOTIONS).to_string())

    fig_cm, axes_cm = plt.subplots(1, len(cm_pairs), figsize=(6.5 * len(cm_pairs), 5),
                                    squeeze=False)
    for col, (title, dfm) in enumerate(cm_pairs):
        cm = confusion_matrix(dfm["emotion_true"], dfm["ext_pred"], labels=list(range(NUM_EMO)))
        cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        ax = axes_cm[0, col]
        sns.heatmap(cm_norm, annot=cm, fmt="d",
                    cmap="Blues", vmin=0, vmax=1,
                    xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax,
                    cbar=(col == len(cm_pairs) - 1))
        ax.set_xlabel(f"External ({ext_name}) prediction")
        ax.set_ylabel("Ground truth")
        ax.set_title(f"{title} — row-normalized (test)")
    plt.tight_layout()
    plt.show()

## Optional: dimensional audio check (MSP-Podcast wav2vec2)

Second external perspective — valence/arousal on the *original* audio. Used only to confirm that our test-split emotion labels align with what an independent audio model perceives. Not used to score generation (generation is video-only).

In [28]:
RUN_AUDIO_SANITY = True

if RUN_AUDIO_SANITY:
    # The audeering MSP-DIM model uses a custom regression head that AutoModel does NOT load
    # (it would silently return base wav2vec2 hidden states instead of arousal/dominance/valence).
    # Per the model card, we have to define the head and wrap a Wav2Vec2Model ourselves.
    from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2PreTrainedModel

    EXT_AUDIO_MODEL = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

    class _RegressionHead(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.dense = nn.Linear(config.hidden_size, config.hidden_size)
            self.dropout = nn.Dropout(config.final_dropout)
            self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

        def forward(self, x):
            x = self.dropout(x)
            x = self.dense(x)
            x = torch.tanh(x)
            x = self.dropout(x)
            return self.out_proj(x)

    class _AudeeringEmotionModel(Wav2Vec2PreTrainedModel):
        """Mirror of the official model card snippet. Output `logits` = (arousal, dominance, valence) in [0, 1]."""

        def __init__(self, config):
            super().__init__(config)
            self.config = config
            self.wav2vec2 = Wav2Vec2Model(config)
            self.classifier = _RegressionHead(config)
            self.init_weights()

        def forward(self, input_values, attention_mask=None):
            outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
            hidden = outputs[0].mean(dim=1)
            return self.classifier(hidden)  # (B, 3) — arousal, dominance, valence

    ext_audio_proc = Wav2Vec2Processor.from_pretrained(EXT_AUDIO_MODEL)
    ext_audio_model = _AudeeringEmotionModel.from_pretrained(EXT_AUDIO_MODEL).to(DEVICE).eval()
    for p in ext_audio_model.parameters():
        p.requires_grad = False
    print(f"Loaded MSP-DIM regressor: out_dim={ext_audio_model.config.num_labels} (arousal, dominance, valence)")

    @torch.no_grad()
    def audio_vad(wav_1d):
        # Audeering model expects 16 kHz mono; processor handles normalization.
        inputs = ext_audio_proc(wav_1d.numpy(), sampling_rate=SR, return_tensors="pt", padding=True)
        kwargs = {"input_values": inputs["input_values"].to(DEVICE)}
        if "attention_mask" in inputs:
            kwargs["attention_mask"] = inputs["attention_mask"].to(DEVICE)
        out = ext_audio_model(**kwargs)  # (1, 3)
        return out.squeeze(0).cpu().numpy()  # arousal, dominance, valence in [0, 1]

    rows = []
    for s in tqdm(test_ds, desc="External audio VAD"):
        a, d, val = audio_vad(s["audio"])
        rows.append({"emotion": EMOTIONS[s["emotion"]],
                     "arousal": float(a),
                     "dominance": float(d),
                     "valence": float(val)})
    vad_df = pd.DataFrame(rows)
    print("\n=== Ground-truth audio VAD (MSP-Podcast model, scores in [0, 1]) ===")
    print(vad_df.groupby("emotion")[["arousal", "dominance", "valence"]].mean().round(3))
    print("\nExpected (if labels agree with perception):")
    print("  happy     — high valence, mid-high arousal")
    print("  sad       — low valence, low arousal")
    print("  angry     — low valence, high arousal, high dominance")
    print("  disgust   — low valence, mid-low arousal")
    vad_df.to_csv(OUT_DIR / "audio_vad.csv", index=False)

    del ext_audio_model
    torch.cuda.empty_cache()

Loaded MSP-DIM regressor: out_dim=3 (arousal, dominance, valence)


External audio VAD: 100%|██████████| 96/96 [00:22<00:00,  4.32it/s]


=== Ground-truth audio VAD (MSP-Podcast model, scores in [0, 1]) ===
         arousal  dominance  valence
emotion                             
angry      0.753      0.775    0.251
disgust    0.573      0.631    0.294
happy      0.627      0.659    0.345
sad        0.439      0.521    0.353

Expected (if labels agree with perception):
  happy     — high valence, mid-high arousal
  sad       — low valence, low arousal
  angry     — low valence, high arousal, high dominance
  disgust   — low valence, mid-low arousal


## Thesis-ready combined table

Internal val F1 (from `04a` ablation cell + `06` finetune cell) joined with external F1 from BOTH classifiers in this notebook (`dima806` + `trpakov`). Side-by-side answer to: *do the gains transfer beyond the encoder used in training, and do TWO independent observers agree?*

Cross-classifier agreement is the strongest signal for thesis: a config with positive Δ on both externals is a real win; positive on one + negative on the other = classifier-specific bias, not a real effect.

Fill `INTERNAL_VAL_F1` below from `04a` cell 14 (best_f1) and `06` cell 8 (best_f1) once those notebooks are re-run with 4 emotions.

In [ ]:
# Internal val F1 — filled from 04a cell 14 (best_f1) and 06 cell 8 (best_f1)
# after 4-emo runs (2026-05-02). Keys must match CHECKPOINTS names in cell 4.
INTERNAL_VAL_F1 = {
    # 04a ablation (sweep over loss types at fixed weights)
    "abl-baseline": 0.619,
    "abl-cos-only": 0.574,
    "abl-ce-only":  0.696,
    "abl-ce-cos":   0.655,
    "abl-ce-kl":    0.737,
    # 06 weight sweep (CE+KL family, multiple scales)
    "baseline":     0.624,
    "cekl-01":      0.742,
    "cekl-025":     0.691,
    "cekl-04":      0.739,
    "cekl-05":      0.736,
}

# Build combined table: one row per config, columns include internal F1 + per-external F1.
ext_names = list(EXTERNALS)
rows = []
for name, _ in CHECKPOINTS:
    row = {"config": name, "internal_val_F1": INTERNAL_VAL_F1.get(name)}
    for ext in ext_names:
        sdf = summary_per_ext[ext]
        sub = sdf.loc[sdf["config"] == name]
        if len(sub):
            row[f"ext_F1_{ext}"]    = float(sub["ext_F1_macro"].iloc[0])
            row[f"Δ_F1_{ext}"]      = float(sub["Δ_F1_vs_baseline"].iloc[0])
            row[f"ext_acc_{ext}"]   = float(sub["ext_accuracy"].iloc[0])
    rows.append(row)
combined = pd.DataFrame(rows)

# Highlight cross-external agreement: average ext F1 + flag for both-positive
ext_cols = [f"Δ_F1_{ext}" for ext in ext_names]
combined["mean_Δ_F1"]      = combined[ext_cols].mean(axis=1)
combined["both_positive"]  = (combined[ext_cols] > 0).all(axis=1)
combined["both_negative"]  = (combined[ext_cols] < 0).all(axis=1)
combined = combined.sort_values("mean_Δ_F1", ascending=False).reset_index(drop=True)

print("\n=== Internal vs External F1 (for thesis table) ===")
print(combined.to_string(index=False))
combined.to_csv(OUT_DIR / "internal_vs_external.csv", index=False)

## Val-split external eval — internal vs external on the SAME split

Test-split results above answer *generalization*. This block re-runs the
two externals on the **val** split — the same split used to select
checkpoints in `04a` and `06` via internal F1. If internal val F1 and
external val F1 disagree on this exact split, it is direct evidence of
circularity in checkpoint selection (we picked the model that fooled
the internal classifier on val, not the model that an independent
observer would have picked).

Compute: ~Wav2Lip generation × 96 val samples × 10 checkpoints × 2 externals
(roughly the same time as the test-split block in cells 13–15).


In [ ]:
# Build val dataset (same class as test, just split="val")
val_ds = Wav2LipTestDataset(METADATA, split="val")
print(f"Val samples: {len(val_ds)}")
from collections import Counter
print("Per-emotion counts:",
      dict(Counter(EMOTIONS[s["emotion"]] for s in val_ds)))


def evaluate_checkpoint_on(name, ckpt_path, ds, tag):
    """Same logic as evaluate_checkpoint but on an explicit dataset.
    `tag` distinguishes saved JSONs (val vs test)."""
    model = load_wav2lip(ckpt_path, DEVICE)
    rows_per_ext = {ext_name: [] for ext_name in EXTERNALS}
    for i in tqdm(range(len(ds)), desc=f"[{name}/{tag}]", leave=False):
        sample = ds[i]
        gen = generate_full_video(model, sample)
        for ext_name in EXTERNALS:
            mean_probs, sample_label, per_frame = external_predict_video(gen, ext_name)
            rows_per_ext[ext_name].append({
                "sample_id":     sample["sample_id"],
                "emotion_true":  sample["emotion"],
                "ext_pred":      sample_label,
                "ext_probs":     mean_probs.tolist(),
                "ext_per_frame": per_frame,
            })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    out = {}
    for ext_name, rows in rows_per_ext.items():
        df = pd.DataFrame(rows)
        df.to_json(OUT_DIR / f"{name}__{ext_name}__{tag}.json", orient="records")
        out[ext_name] = df
    return out


per_model_val = {}
for name, path in CHECKPOINTS:
    print(f"\n=== Evaluating {name} on VAL ===")
    per_model_val[name] = evaluate_checkpoint_on(name, path, val_ds, tag="val")
print("\nVal eval done. ", list(EXTERNALS), "externals on", len(per_model_val), "checkpoints.")


In [ ]:
# Per-external summary on VAL (mirrors cell 14 / summary_per_ext but for val).
summary_per_ext_val = {}
for ext_name in EXTERNALS:
    rows = []
    for name in [n for n, _ in CHECKPOINTS]:
        acc, f1m, per = model_metrics(per_model_val[name][ext_name])
        rows.append({
            "config": name,
            "ext_val_acc": acc,
            "ext_val_F1_macro": f1m,
            **{f"ext_val_F1_{EMOTIONS[i]}": per[i] for i in range(NUM_EMO)},
        })
    df = pd.DataFrame(rows)
    base = df.loc[df["config"] == "baseline", "ext_val_F1_macro"].iloc[0]
    df["Δ_val_F1_vs_baseline"] = df["ext_val_F1_macro"] - base
    df = df.sort_values("ext_val_F1_macro", ascending=False).reset_index(drop=True)
    summary_per_ext_val[ext_name] = df
    print(f"\n=== EXTERNAL VAL ({EXTERNALS[ext_name]['model_id']}) ===")
    print(df.to_string(index=False))
    df.to_csv(OUT_DIR / f"summary_val__{ext_name}.csv", index=False)


In [ ]:
# Internal val F1 vs external val F1 vs external test F1 — three-way thesis table.
# Reveals (a) circularity within val (int vs ext on same split) and
# (b) val→test transfer (ext_val vs ext_test).
rows = []
for name, _ in CHECKPOINTS:
    row = {"config": name, "internal_val_F1": INTERNAL_VAL_F1.get(name)}
    for ext in EXTERNALS:
        s_test = summary_per_ext[ext]
        s_val  = summary_per_ext_val[ext]
        sub_t = s_test.loc[s_test["config"] == name]
        sub_v = s_val.loc[s_val["config"] == name]
        if len(sub_t):
            row[f"ext_test_F1_{ext}"] = float(sub_t["ext_F1_macro"].iloc[0])
            row[f"Δ_test_{ext}"]      = float(sub_t["Δ_F1_vs_baseline"].iloc[0])
        if len(sub_v):
            row[f"ext_val_F1_{ext}"]  = float(sub_v["ext_val_F1_macro"].iloc[0])
            row[f"Δ_val_{ext}"]       = float(sub_v["Δ_val_F1_vs_baseline"].iloc[0])
    rows.append(row)

three_way = pd.DataFrame(rows)
val_delta_cols  = [f"Δ_val_{ext}"  for ext in EXTERNALS]
test_delta_cols = [f"Δ_test_{ext}" for ext in EXTERNALS]
three_way["mean_Δ_val"]  = three_way[val_delta_cols].mean(axis=1)
three_way["mean_Δ_test"] = three_way[test_delta_cols].mean(axis=1)
three_way = three_way.sort_values("mean_Δ_val", ascending=False).reset_index(drop=True)

print("\n=== THREE-WAY TABLE: internal val | external val | external test ===")
print(three_way.to_string(index=False))
three_way.to_csv(OUT_DIR / "three_way_int_val_test.csv", index=False)

# Quick sanity diagnostics
import scipy.stats as sstats
print("\n--- Diagnostic correlations across configs ---")
for ext in EXTERNALS:
    iv = three_way["internal_val_F1"]
    ev = three_way[f"ext_val_F1_{ext}"]
    et = three_way[f"ext_test_F1_{ext}"]
    rho_iv_ev, p_iv_ev = sstats.spearmanr(iv, ev, nan_policy="omit")
    rho_ev_et, p_ev_et = sstats.spearmanr(ev, et, nan_policy="omit")
    rho_iv_et, p_iv_et = sstats.spearmanr(iv, et, nan_policy="omit")
    print(f"  {ext}: ρ(int_val, ext_val)={rho_iv_ev:.3f} (p={p_iv_ev:.3f}) | "
          f"ρ(ext_val, ext_test)={rho_ev_et:.3f} (p={p_ev_et:.3f}) | "
          f"ρ(int_val, ext_test)={rho_iv_et:.3f} (p={p_iv_et:.3f})")
print("\nReading guide:")
print("  ρ(int_val, ext_val) < 0  → checkpoint selection by internal F1 picks models that")
print("                             an external observer would NOT pick (circularity).")
print("  ρ(ext_val, ext_test) > 0 → external eval generalizes val→test (eval is consistent).")
print("  ρ(int_val, ext_test) < 0 → internal val ranking inverted on independent test.")
